# Timing Python lists versus NumPy

Generate **100,000 random numbers** using a Python list comprehension and NumPy, then compare their runtime with `%timeit`.

Run cells in order with **Shift + Enter**. Timing results will be produced on your own computer; none are pre-filled.


## 1. Load the tools

If importing NumPy fails, run this in a separate notebook cell, then retry the import:

```python
import sys
!uv pip install --python "{sys.executable}" numpy
```

The random generators are seeded outside the timed operations. Their states advance as the benchmarks generate new numbers; they do not produce identical sequences.


In [4]:
import random
import numpy as np

N = 100_000
python_rng = random.Random(42)
numpy_rng = np.random.default_rng(42)

print(f'Numbers per operation: {N:,}')
print('NumPy version:', np.__version__)


Numbers per operation: 100,000
NumPy version: 2.5.4


## 2. Inspect the two results

Python calls its random generator once per loop iteration. NumPy generates a whole array with one call.

Both produce values in **[0, 1)**, but they use different random generators and produce different containers: a Python list and a NumPy array. This is a practical workflow comparison, not a controlled test of loop syntax alone.


In [5]:
python_values = [python_rng.random() for _ in range(N)]
numpy_values = numpy_rng.random(N)

print('Python:', type(python_values).__name__, 'length:', len(python_values))
print('NumPy:', type(numpy_values).__name__, 'shape:', numpy_values.shape)
print('First three Python values:', python_values[:3])
print('First three NumPy values:', numpy_values[:3])


Python: list length: 100000
NumPy: ndarray shape: (100000,)
First three Python values: [0.6394267984578837, 0.025010755222666936, 0.27502931836911926]
First three NumPy values: [0.77395605 0.43887844 0.85859792]


## 3. Time the list comprehension

`%timeit` repeats the operation and reports time **per complete batch of 100,000 numbers**, not time per number. The options below use 10 batches per repeat and 5 repeats.

Assigning the timing report with `-o` lets us compare the measured averages afterward.


In [6]:
list_timing = %timeit -o -n 10 -r 5 [python_rng.random() for _ in range(N)]


4.44 ms ± 118 μs per loop (mean ± std. dev. of 5 runs, 10 loops each)


## 4. Time NumPy

Use the same batch size and timing options.


In [7]:
numpy_timing = %timeit -o -n 10 -r 5 numpy_rng.random(N)


490 μs ± 64.4 μs per loop (mean ± std. dev. of 5 runs, 10 loops each)


## 5. Compare your measurements

NumPy will often be faster because generation occurs in compiled code and values are stored in a compact numeric array. Actual results depend on your hardware, package versions, and other running work.


In [8]:
list_ms = list_timing.average * 1_000
numpy_ms = numpy_timing.average * 1_000
ratio = list_timing.average / numpy_timing.average

print(f'Python list: {list_ms:.3f} ms per 100,000 numbers')
print(f'NumPy array: {numpy_ms:.3f} ms per 100,000 numbers')
print(f'List time / NumPy time: {ratio:.2f}')
if ratio >= 1:
    print(f'NumPy used about {1 / ratio:.1%} of the list-comprehension time.')
else:
    print('NumPy took longer in this run; repeat under similar system load.')


Python list: 4.442 ms per 100,000 numbers
NumPy array: 0.490 ms per 100,000 numbers
List time / NumPy time: 9.07
NumPy used about 11.0% of the list-comprehension time.


## 6. Optional: require both outputs to be NumPy arrays

The original benchmarks return different containers. If your task specifically needs an array, include list-to-array conversion in the Python workflow. The conversion cost is part of this extra benchmark; the RNG algorithms still differ.


In [9]:
converted_timing = %timeit -o -n 10 -r 5 np.array([python_rng.random() for _ in range(N)])

print(f'List comprehension + conversion: {converted_timing.average * 1_000:.3f} ms')
print(f'Direct NumPy array: {numpy_timing.average * 1_000:.3f} ms')


6.53 ms ± 452 μs per loop (mean ± std. dev. of 5 runs, 10 loops each)
List comprehension + conversion: 6.528 ms
Direct NumPy array: 0.490 ms


## Record what you learned

1. Which method was faster, and what was the time ratio?
2. Does the reported time describe one random number or the entire batch?
3. What changed when you included conversion to an array?
4. Why does this comparison involve more than just Python loops?

**My observations:**

_Write your timing results and explanation here._

Before finishing, save with **⌘S**. To check that your notebook works from a clean start, restart its kernel and run all cells.
